# TME Spatial pipeline demo

This notebook shows how to call the refactored Python modules directly.

It uses the same core logic as the Streamlit app, but without the Streamlit UI.  
Run from the repo root, or let the first code cell add the repo root to `sys.path`.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
from pathlib import Path

from src.tme_spatial.models import ChannelConfig, PipelineConfig, NucleiParams, RegionParams
from src.tme_spatial.io import files_to_long_df
from src.tme_spatial.visualization import overlay_multi_channels, plot_split_channels
from src.tme_spatial.nuclei_segmentation import run_nuclei_segmentation
from src.tme_spatial.celltype_assignment import run_celltype_assignment

# --- edit these values for your dataset ---
FOLDER = Path("data/demo1")          # change to your folder
PIXEL_SIZE_UM = (0.5, 0.5)           # change to your pixel size
CHANNELS = [
    ChannelConfig(file="DAPI.csv", channel="DAPI", color_hex="#ffffff"),
    ChannelConfig(file="CD3.csv", channel="CD3", color_hex="#00ff00"),
    ChannelConfig(file="PDL1.csv", channel="PDL1", color_hex="#ff00ff"),
]
SAVE_DIR = FOLDER / "outs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

config = PipelineConfig(
    folder=FOLDER,
    save_dir=SAVE_DIR,
    pixel_size_um=PIXEL_SIZE_UM,
    channels=CHANNELS,
    overlay_channels=[c.channel for c in CHANNELS],
)

config.to_json_dict()

In [ ]:
df_pixels, shapes = files_to_long_df(
    folder=config.folder,
    channels_cfg=[c.to_dict() for c in config.channels],
    image_id=config.image_id,
    pixel_size_um=config.pixel_size_um,
)

overlay_fig, _ = overlay_multi_channels(
    df=df_pixels,
    shapes=shapes,
    image_id=config.image_id,
    channels_cfg=[c.to_dict() for c in config.channels],
    overlay_channels=config.overlay_channels,
    white_channel=config.white_channel,
    white_weight=config.white_weight,
    pixel_size_um=config.pixel_size_um,
    save_path=config.save_dir / "overlay.svg",
)

split_fig = plot_split_channels(
    df=df_pixels,
    shapes=shapes,
    image_id=config.image_id,
    channels_cfg=[c.to_dict() for c in config.channels],
    pixel_size_um=config.pixel_size_um,
    save_path=config.save_dir / "split_channels.svg",
)

overlay_fig

In [ ]:
nuclei_params = NucleiParams(nucleus_channel="DAPI")

nuclei_result = run_nuclei_segmentation(
    df_pixels=df_pixels,
    shapes=shapes,
    image_id=config.image_id,
    save_dir=config.save_dir,
    pixel_size_um=config.pixel_size_um,
    params=nuclei_params,
    save_outputs=True,
)

nuclei_result["df_props"].head()

In [ ]:
CELLTYPE_CFG = [
    {
        "name": "double_positive",
        "color_hex": "#ff0000",
        "mode": "simple",
        "all_pos": ["CD3", "PDL1"],
        "all_neg": [],
        "any_pos_groups": [],
    },
    {
        "name": "CD3_only",
        "color_hex": "#00ff00",
        "mode": "simple",
        "all_pos": ["CD3"],
        "all_neg": ["PDL1"],
        "any_pos_groups": [],
    },
    {
        "name": "fallback_other",
        "color_hex": "#00ffff",
        "mode": "simple",
        "all_pos": [],
        "all_neg": [],
        "any_pos_groups": [],
    },
]

assignment_result = run_celltype_assignment(
    folder=config.folder,
    save_dir=config.save_dir,
    pixel_size_um=config.pixel_size_um,
    image_id=config.image_id,
    channels_cfg=[c.to_dict() for c in config.channels],
    celltype_cfg=CELLTYPE_CFG,
    labels=nuclei_result["labels"],
    df_pixels=df_pixels,
    shapes=shapes,
    save_outputs=True,
)

assignment_result["counts"]